# Blip2 Opt 2 7b COCO Baseline

This notebook was reorganized for the GitHub reproducibility package.
Original file: `COCO-Baseline/BLIP-2_OPT-2.7b.ipynb`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


In [ ]:
# Kurulum ve Kütüphaneler

# Gerekli kütüphaneler

!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate bitsandbytes
!pip install -q pycocotools
!pip install -q git+https://github.com/salaniz/pycocoevalcap

from google.colab import drive
import os
import torch

# Drive Bağlantısı
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Hugging Face Login (BLIP-2 public olduğu için zorunlu değil ama standart olsun)
from huggingface_hub import login
login()  # Enter your Hugging Face token interactively; never commit tokens. # İstersen token ekleyebilirsin

In [ ]:
# Modeli Yükleme (BLIP-2 OPT 2.7B)

from transformers import Blip2Processor, Blip2ForConditionalGeneration

MODEL_ID = "Salesforce/blip2-opt-2.7b"
print(f"⏳ {MODEL_ID} yükleniyor (2.7B Parametre)...")

# İşlemciyi Yükle
processor = Blip2Processor.from_pretrained(MODEL_ID)

# Modeli Yükle (A100 için float16 ve auto device map)
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
).eval()

print("✅ Model Hazır! (Encoder: ViT, Decoder: OPT-2.7B - Decoder Only)")

In [ ]:
# Dosya Yolları ve Kontrol (PaliGemma ile Birebir Aynı)

import os
import json

# ==========================================
# AYARLAR
# ==========================================
COCO_ROOT = "/content/drive/MyDrive/datasets/coco2014"
KARPATHY_TEST = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json"

print(f"📂 JSON Dosyası: {KARPATHY_TEST}")
print(f"📂 Resim Kökü : {COCO_ROOT}")

# JSON Yükle
try:
    with open(KARPATHY_TEST, "r") as f:
        items = json.load(f)
    print(f"✅ JSON başarıyla okundu. Toplam kayıt: {len(items)}")
except Exception as e:
    print(f"❌ JSON okunamadı! Yol yanlış olabilir. Hata: {e}")
    items = []

# Basit Yol Kontrolü (İlk 5 resim)
print("\n🚀 Yol Testi (İlk 5 resim)...")
for i, item in enumerate(items[:5]):
    img_rel = item["image"]
    full_path = os.path.join(COCO_ROOT, img_rel)
    if os.path.exists(full_path):
        print(f"   ✅ [OK] {full_path}")
    else:
        print(f"   ❌ [YOK] {full_path}")

In [ ]:
# Tahmin (Inference) Döngüsü

from PIL import Image
import torch
import re
import os
import json

# Çıktı Dosyası
OUTPUT_FILE = "blip2_opt_2.7b_preds.json"

# ID Parse Fonksiyonu
def coco_id_from_relpath(rel_path: str) -> int:
    m = re.search(r"_(\d{12})\.jpg$", rel_path)
    if not m: return int(rel_path.split('_')[-1].split('.')[0])
    return int(m.group(1))

preds = []
print(f"🚀 {len(items)} resim için BLIP-2 (OPT) Testi Başlıyor...")

for i, item in enumerate(items):
    image_id = "Bilinmiyor" # Hata durumunda loglayabilmek için
    try:
        # Resim Yolu
        img_rel = item["image"]
        img_path = os.path.join(COCO_ROOT, img_rel)
        image_id = coco_id_from_relpath(img_rel)

        # Resmi Yükle
        image = Image.open(img_path).convert('RGB')

        # BLIP-2 Girdisi (Orjinalindeki gibi promptsuz, saf caption modu)
        inputs = processor(images=image, return_tensors="pt").to("cuda", torch.float16)

        # Üretim
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=30, # COCO captionları genelde kısadır
                do_sample=False,   # Deterministik (Tez için şart)
                num_beams=5        # Kaliteyi artırır
            )

        # Çözme (Decoding)
        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

        preds.append({
            "image_id": image_id,
            "caption": caption
        })

    except Exception as e:
        print(f"⚠️ Hata (ID: {image_id}): {e}")
        # Hata olsa bile listeye ekle ki değerlendirmede ID kayması olmasın
        preds.append({"image_id": image_id, "caption": "error"})

    # İlerleme (Her 100 resimde bir)
    if i % 100 == 0:
        print(f"[{i}/{len(items)}] {caption}")

# Kaydet
with open(OUTPUT_FILE, "w") as f:
    json.dump(preds, f)

print(f"\n✅ İşlem bitti! Tahminler kaydedildi: {OUTPUT_FILE}")

In [ ]:
# Değerlendirme (Java'sız Güvenli Mod)

from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
import json

# Yollar
GT_PATH = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test_gt.json"
PREDS_FILE = "blip2_opt_2.7b_preds.json"

print("📊 Değerlendirme Başlıyor (Java Gerektirmeyen Mod)...")

try:
    coco = COCO(GT_PATH)
    cocoRes = coco.loadRes(PREDS_FILE)
    cocoEval = COCOEvalCap(coco, cocoRes)

    # ID Eşleşmesi
    imgIds = sorted([img['image_id'] for img in json.load(open(PREDS_FILE))])
    cocoEval.params["image_id"] = imgIds

    # Sadece güvenli metrikler (SPICE ve METEOR yok)
    cocoEval.scorers = [
        (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
        (Rouge(), "ROUGE_L"),
        (Cider(), "CIDEr")
    ]

    cocoEval.evaluate()

    print("\n" + "="*40)
    print("🏆 BLIP-2 (OPT-2.7B) SONUÇLARI")
    print("="*40)
    print(f"CIDEr    : {cocoEval.eval['CIDEr']:.3f}")
    print(f"ROUGE-L  : {cocoEval.eval['ROUGE_L']:.3f}")
    print(f"BLEU-4   : {cocoEval.eval['Bleu_4']:.3f}")
    print("="*40)

except Exception as e:
    print(f"⚠️ Hata: {e}")

In [ ]:
# Sonuçları Drive'a Kaydedelim

import shutil
import os
import json

# Klasör Yapısı
SAVE_DIR = "/content/drive/MyDrive/tez_sonuclar/blip2_opt_2.7b"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Tahminleri Yedekle
SOURCE_PREDS = "blip2_opt_2.7b_preds.json"
TARGET_PREDS = os.path.join(SAVE_DIR, "blip2_opt_2.7b_preds_final.json")
shutil.copy(SOURCE_PREDS, TARGET_PREDS)

# 2. Skorları Kaydet
metrics = {
    "Model": "BLIP-2 (OPT-2.7b)",
    "Backbone": "Decoder-only LLM (OPT)",
    "Scores": {
        "CIDEr": cocoEval.eval['CIDEr'],
        "BLEU_4": cocoEval.eval['Bleu_4'],
        "ROUGE_L": cocoEval.eval['ROUGE_L']
    }
}
with open(os.path.join(SAVE_DIR, "blip2_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

# 3. TXT Raporu
with open(os.path.join(SAVE_DIR, "blip2_scores.txt"), "w") as f:
    f.write(f"=== {metrics['Model']} ===\n")
    f.write(f"Backbone: {metrics['Backbone']}\n")
    f.write("-" * 20 + "\n")
    for k, v in metrics['Scores'].items():
        f.write(f"{k:<10}: {v:.3f}\n")

print(f"✅ Tüm sonuçlar kaydedildi: {SAVE_DIR}")

In [ ]:
# Parametre Sayısı

# Model Parametrelerini Say
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*30)
print(f"🧠 MODEL: BLIP-2 (OPT-2.7B)")
print("="*30)
print(f"Toplam Parametre: {total_params:,}")
print(f"Toplam (Milyar) : {total_params / 1e9:.3f} B")
print("-" * 30)
print(f"Eğitilebilir    : {trainable_params:,}")
print("="*30)